<a href="https://colab.research.google.com/github/RayanMohammed/de-pipeline/blob/main/FHIR_data_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install supabase duckdb pydantic pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!unzip /content/drive/MyDrive/data/FHIR_data/data.zip -d /content/patients_data

Archive:  /content/drive/MyDrive/data/FHIR_data/data.zip
  inflating: /content/patients_data/Sharolyn456_Huels583_5b24c87b-6223-f5b4-51e9-82051159bd1d.json  
  inflating: /content/patients_data/Warner493_McKenzie376_13472219-c176-990a-641f-14cf9d4d8480.json  
  inflating: /content/patients_data/Terry864_Glover433_d20a36fc-23ba-8462-bf39-864000fbf25f.json  
  inflating: /content/patients_data/Georgiann138_Dickinson688_53e4891a-9108-67ef-d973-3b6a98404249.json  
  inflating: /content/patients_data/Dominic463_Parker433_03a7cc66-36c5-356c-419f-93bfbd7b558d.json  
  inflating: /content/patients_data/Gertrudis163_Tasia358_Paucek755_a8286256-4ef3-c614-4371-085d7c8cfcb2.json  
  inflating: /content/patients_data/Latrina689_Hilpert278_34b8aa06-9904-df2f-e571-c6c5e767f24b.json  
  inflating: /content/patients_data/Tuan998_Bernhard322_68988c92-ff15-280a-9b96-5d8f72e7b0ef.json  
  inflating: /content/patients_data/Harvey63_Ernser583_3ab40d52-44be-ae07-2177-2875cc65f5a8.json  
  inflating: /content

In [4]:
import glob
import json
import pandas as pd
from typing import Optional
from pydantic import BaseModel
from google.colab import userdata
from supabase import create_client

url = userdata.get("SUPABASE_URL")
key = userdata.get("SUPABASE_KEY")
supabase = create_client(url, key)

class PatientResponse(BaseModel):
    id: str
    gender: str | None = None
    birth_date: str | None = None
    height_cm: float | None = None
    weight_kg: float | None = None
    bmi: float | None = None
    bmi_category: str | None = None

def classify_bmi(bmi_val: float | None):
    if bmi_val is None:
        return None
    if bmi_val < 18.5:
        return "Underweight"
    elif 18.5 <= bmi_val < 25.0:
        return "Normal"
    elif 25.0 <= bmi_val < 30.0:
        return "Overweight"
    else:
        return "Obese"

def extract_clinical_data(bundle_dict: dict):
    if bundle_dict.get('resourceType') != 'Bundle' or 'entry' not in bundle_dict:
        return None

    first_resource = bundle_dict['entry'][0].get('resource', {})
    if first_resource.get('resourceType') != 'Patient':
        return None

    patient_info = {}
    for entry in bundle_dict['entry']:
        resource = entry.get('resource')
        if resource:
            if resource.get('resourceType') == 'Patient':
                patient_info['id'] = resource.get('id')
                patient_info['gender'] = resource.get('gender')
                patient_info['birth_date'] = resource.get('birthDate')
            elif resource.get('resourceType') == 'Observation':
                codes = resource.get('code', {}).get('coding', [])
                for code in codes:
                    if code.get('code') == '8302-2': #height in cm
                        patient_info['height_cm'] = resource.get('valueQuantity', {}).get('value')
                    elif code.get('code') == '29463-7': #weight in kg
                        patient_info['weight_kg'] = resource.get('valueQuantity', {}).get('value')

    #bmi column creation
    if 'height_cm' in patient_info and 'weight_kg' in patient_info:
        bmi_calc = round(patient_info['weight_kg'] / ((patient_info['height_cm'] / 100) ** 2), 1)
        patient_info['bmi'] = bmi_calc
        patient_info['bmi_category'] = classify_bmi(bmi_calc)
    else:
        patient_info['bmi'] = None
        patient_info['bmi_category'] = None

    if 'id' in patient_info:
        return PatientResponse(**patient_info).model_dump()

    return None

In [5]:
full_list = glob.glob('/content/patients_data/*.json')
valid_patients = []

for item in full_list:
    try:
        with open(item, 'r') as file:
            fhir_bundle = json.load(file)

        parsed_record = extract_clinical_data(fhir_bundle)
        if parsed_record:
            valid_patients.append(parsed_record)

    except Exception as e:
        pass

print(f"Extracted {len(valid_patients)} patient structures.")

Extracted 109 patient structures.


In [6]:
df = pd.DataFrame(valid_patients)

df['id'] = df['id'].astype(str)
if 'birth_date' in df.columns:
    df['birth_date'] = df['birth_date'].astype(str)

df = df.dropna(subset=['bmi'])
df = df.drop_duplicates(subset=['id'])

print(f"Number of clean records: {len(df)}")

Number of clean records: 109


In [8]:
#parquet file
parquet_destination = '/content/drive/MyDrive/data/patients_clean.parquet'
df.to_parquet(parquet_destination, engine='pyarrow', compression='snappy', index=False)
print(f"Archived snapshot in Parquet to: {parquet_destination}")

#upload to supabase
records = df.to_dict(orient="records")
supabase.table('patients').upsert(records).execute()
print(f"Successfully synced {len(records)} verified records to Supabase PostgreSQL.")

Archived snapshot in Parquet to: /content/drive/MyDrive/data/patients_clean.parquet
Successfully synced 109 verified records to Supabase PostgreSQL.
